# NSAgent — A Simple Agentic Framework

A lightweight agent that works through three **blended** phases for any task:

| Phase | Purpose |
|---|---|
| **Gather Context** | Understand the task, read files, browse pages, inspect state |
| **Take Action** | Execute tools, call APIs, write files, run commands |
| **Verify Results** | Check outputs, validate correctness, decide if done or loop back |

These phases blend naturally — the agent decides when to gather more context, act, or verify.

---

## 1. Setup & Configuration

In [ ]:
# Install dependencies (run once)
# !pip install openai httpx mcp

In [ ]:
import os
import json
import subprocess
import inspect
import textwrap
import traceback
import httpx
from typing import Any, Callable, Optional
from dataclasses import dataclass, field

# Configuration — set your API key and model
CONFIG = {
    "api_key": os.environ.get("OPENAI_API_KEY", "your-api-key-here"),
    "base_url": os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1"),
    "model": os.environ.get("OPENAI_MODEL", "gpt-4o"),
    "max_iterations": 15,
    "temperature": 0.2,
}

print(f"Model: {CONFIG['model']}")
print(f"Base URL: {CONFIG['base_url']}")
print(f"Max iterations: {CONFIG['max_iterations']}")

## 2. Tool & Skill Registry

Tools are simple functions the agent can call. Skills are higher-level capabilities that may orchestrate multiple tools. Both are registered via decorators and are fully expandable.

In [ ]:
@dataclass
class ToolDef:
    """Definition of a registered tool or skill."""
    name: str
    description: str
    parameters: dict  # JSON Schema
    fn: Callable
    is_skill: bool = False


def _python_type_to_json(annotation) -> str:
    """Map Python type annotations to JSON Schema types."""
    mapping = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array", dict: "object"}
    return mapping.get(annotation, "string")


def _build_schema(fn: Callable) -> dict:
    """Auto-generate a JSON Schema for a function's parameters."""
    sig = inspect.signature(fn)
    properties = {}
    required = []
    for name, param in sig.parameters.items():
        ptype = _python_type_to_json(param.annotation) if param.annotation != inspect.Parameter.empty else "string"
        properties[name] = {"type": ptype, "description": f"Parameter: {name}"}
        if param.default is inspect.Parameter.empty:
            required.append(name)
    return {"type": "object", "properties": properties, "required": required}


class ToolRegistry:
    """Registry for tools and skills. Supports decorator-based registration."""

    def __init__(self):
        self._tools: dict[str, ToolDef] = {}

    def tool(self, name: str = None, description: str = "No description"):
        """Decorator to register a tool."""
        def decorator(fn: Callable):
            tool_name = name or fn.__name__
            self._tools[tool_name] = ToolDef(
                name=tool_name, description=description,
                parameters=_build_schema(fn), fn=fn, is_skill=False
            )
            return fn
        return decorator

    def skill(self, name: str = None, description: str = "No description"):
        """Decorator to register a skill (higher-level capability)."""
        def decorator(fn: Callable):
            skill_name = name or fn.__name__
            self._tools[skill_name] = ToolDef(
                name=skill_name, description=description,
                parameters=_build_schema(fn), fn=fn, is_skill=True
            )
            return fn
        return decorator

    def register(self, name: str, description: str, parameters: dict, fn: Callable, is_skill=False):
        """Programmatic registration (for MCP tools, etc.)."""
        self._tools[name] = ToolDef(name=name, description=description, parameters=parameters, fn=fn, is_skill=is_skill)

    def get(self, name: str) -> Optional[ToolDef]:
        return self._tools.get(name)

    def list_tools(self) -> list[ToolDef]:
        return list(self._tools.values())

    def to_openai_tools(self) -> list[dict]:
        """Convert registry to OpenAI function-calling tool format."""
        return [
            {
                "type": "function",
                "function": {
                    "name": t.name,
                    "description": t.description,
                    "parameters": t.parameters,
                }
            }
            for t in self._tools.values()
        ]

    def call(self, name: str, arguments: dict) -> str:
        """Call a tool by name with the given arguments."""
        tool = self._tools.get(name)
        if not tool:
            return f"Error: Unknown tool '{name}'"
        try:
            result = tool.fn(**arguments)
            return str(result) if result is not None else "(no output)"
        except Exception as e:
            return f"Error calling {name}: {e}\n{traceback.format_exc()}"


registry = ToolRegistry()
print("Tool registry initialized.")

## 3. Built-in Tools (Out of the Box)

These tools are available immediately without any configuration.

In [ ]:
# --- File I/O ---

@registry.tool(name="read_file", description="Read the contents of a file at the given path.")
def read_file(path: str) -> str:
    with open(path, "r", encoding="utf-8") as f:
        return f.read()


@registry.tool(name="write_file", description="Write content to a file. Creates or overwrites the file.")
def write_file(path: str, content: str) -> str:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    return f"Wrote {len(content)} chars to {path}"


@registry.tool(name="list_directory", description="List files and folders in a directory.")
def list_directory(path: str = ".") -> str:
    entries = os.listdir(path)
    result = []
    for e in sorted(entries):
        full = os.path.join(path, e)
        kind = "dir" if os.path.isdir(full) else "file"
        result.append(f"  [{kind}] {e}")
    return f"Contents of {path}:\n" + "\n".join(result)


# --- Shell Commands ---

@registry.tool(name="run_command", description="Run a shell command and return stdout/stderr. Use for git, pip, etc.")
def run_command(command: str, cwd: str = ".") -> str:
    try:
        result = subprocess.run(
            command, shell=True, capture_output=True, text=True,
            timeout=60, cwd=cwd
        )
        output = ""
        if result.stdout:
            output += result.stdout
        if result.stderr:
            output += f"\n[stderr] {result.stderr}"
        output += f"\n[exit code: {result.returncode}]"
        return output.strip()
    except subprocess.TimeoutExpired:
        return "Error: command timed out after 60 seconds."


# --- Web Fetch ---

@registry.tool(name="web_fetch", description="Fetch a URL and return the response text (first 10000 chars).")
def web_fetch(url: str) -> str:
    try:
        with httpx.Client(follow_redirects=True, timeout=15) as client:
            r = client.get(url)
            return r.text[:10000]
    except Exception as e:
        return f"Error fetching {url}: {e}"


# --- Python Exec ---

@registry.tool(name="python_exec", description="Execute a Python code snippet and return the output. Use for calculations, data processing, etc.")
def python_exec(code: str) -> str:
    import io, contextlib
    stdout = io.StringIO()
    local_vars = {}
    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, {"__builtins__": __builtins__}, local_vars)
        output = stdout.getvalue()
        return output if output else "(executed successfully, no output)"
    except Exception as e:
        return f"Error: {e}\n{traceback.format_exc()}"


# --- Task Status ---

@registry.tool(name="task_done", description="Call this when the task is fully complete. Provide a summary of what was accomplished.")
def task_done(summary: str) -> str:
    return f"TASK_COMPLETE: {summary}"


print(f"Registered {len(registry.list_tools())} built-in tools:")
for t in registry.list_tools():
    label = "[skill]" if t.is_skill else "[tool]"
    print(f"  {label} {t.name} — {t.description}")

## 4. MCP Server Integration (Playwright Browser Automation)

This section connects to the [Playwright MCP](https://github.com/microsoft/playwright-mcp) server, which provides browser automation tools (navigate, click, type, screenshot, etc.) via the Model Context Protocol.

**Prerequisites:** `npm install -g @playwright/mcp@latest` (or use `npx`)

In [ ]:
import sys
import threading


class MCPClient:
    """
    MCP client using raw subprocess + JSON-RPC over stdin/stdout.
    
    This avoids the MCP SDK's stdio_client which fails in Jupyter
    due to Jupyter replacing sys.stdin/stdout with objects lacking fileno().
    Instead, we spawn the server with subprocess.Popen and speak JSON-RPC directly.
    """

    def __init__(self, server_command: list[str], server_name: str = "mcp"):
        self.server_command = server_command
        self.server_name = server_name
        self._process = None
        self._request_id = 0
        self._tools = {}  # name -> {description, inputSchema}
        self._lock = threading.Lock()

    def _send_request(self, method: str, params: dict = None) -> dict:
        """Send a JSON-RPC request and read the response."""
        self._request_id += 1
        request = {
            "jsonrpc": "2.0",
            "id": self._request_id,
            "method": method,
        }
        if params is not None:
            request["params"] = params

        payload = json.dumps(request)
        # MCP uses Content-Length framed JSON-RPC over stdio
        header = f"Content-Length: {len(payload.encode('utf-8'))}\r\n\r\n"
        with self._lock:
            self._process.stdin.write(header.encode('utf-8'))
            self._process.stdin.write(payload.encode('utf-8'))
            self._process.stdin.flush()
            return self._read_response()

    def _read_response(self) -> dict:
        """Read a Content-Length framed JSON-RPC response, skipping notifications."""
        while True:
            # Read headers
            content_length = None
            while True:
                line = self._process.stdout.readline()
                if not line:
                    raise ConnectionError("MCP server closed the connection.")
                line_str = line.decode('utf-8').strip()
                if line_str == '':
                    break  # End of headers
                if line_str.lower().startswith('content-length:'):
                    content_length = int(line_str.split(':', 1)[1].strip())

            if content_length is None:
                raise ValueError("Missing Content-Length header from MCP server.")

            body = self._process.stdout.read(content_length)
            msg = json.loads(body.decode('utf-8'))

            # Skip JSON-RPC notifications (no 'id' field)
            if 'id' in msg:
                return msg

    def connect(self) -> bool:
        """Start the MCP server subprocess and initialize the session."""
        try:
            self._process = subprocess.Popen(
                self.server_command,
                stdin=subprocess.PIPE,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                # On Windows use CREATE_NO_WINDOW to avoid a console flash
                creationflags=subprocess.CREATE_NO_WINDOW if sys.platform == 'win32' else 0,
            )
        except FileNotFoundError:
            print(f"[MCP:{self.server_name}] Command not found: {self.server_command[0]}")
            print(f"  Make sure it is installed: npm install -g @playwright/mcp@latest")
            return False
        except Exception as e:
            print(f"[MCP:{self.server_name}] Failed to start server: {e}")
            return False

        try:
            # MCP initialize handshake
            init_resp = self._send_request("initialize", {
                "protocolVersion": "2024-11-05",
                "capabilities": {},
                "clientInfo": {"name": "nsagent", "version": "1.0.0"}
            })
            if "error" in init_resp:
                print(f"[MCP:{self.server_name}] Init error: {init_resp['error']}")
                return False

            # Send initialized notification (no id, no response expected)
            notif = json.dumps({"jsonrpc": "2.0", "method": "notifications/initialized"})
            header = f"Content-Length: {len(notif.encode('utf-8'))}\r\n\r\n"
            self._process.stdin.write(header.encode('utf-8'))
            self._process.stdin.write(notif.encode('utf-8'))
            self._process.stdin.flush()

            # Discover tools
            tools_resp = self._send_request("tools/list", {})
            if "result" in tools_resp and "tools" in tools_resp["result"]:
                for tool in tools_resp["result"]["tools"]:
                    self._tools[tool["name"]] = tool

            print(f"[MCP:{self.server_name}] Connected. Discovered {len(self._tools)} tools.")
            return True

        except Exception as e:
            print(f"[MCP:{self.server_name}] Failed to connect: {e}")
            stderr_out = ""
            try:
                stderr_out = self._process.stderr.read(2000).decode('utf-8', errors='ignore')
            except Exception:
                pass
            if stderr_out:
                print(f"  Server stderr: {stderr_out[:300]}")
            self.disconnect()
            return False

    def call_tool(self, name: str, arguments: dict) -> str:
        """Call an MCP tool and return the result as text."""
        if not self._process or self._process.poll() is not None:
            return "Error: MCP server is not running."
        try:
            resp = self._send_request("tools/call", {"name": name, "arguments": arguments})
            if "error" in resp:
                return f"MCP Error: {resp['error']}"
            content = resp.get("result", {}).get("content", [])
            texts = [c.get("text", "") for c in content if c.get("type") == "text"]
            return "\n".join(texts) if texts else "(no output)"
        except Exception as e:
            return f"MCP Error [{name}]: {e}"

    def register_tools(self, tool_registry: ToolRegistry):
        """Register all discovered MCP tools into the agent's tool registry."""
        for name, tool in self._tools.items():
            prefixed_name = f"browser_{name}" if self.server_name == "playwright" else f"{self.server_name}_{name}"
            schema = tool.get("inputSchema", {"type": "object", "properties": {}})
            description = f"[MCP:{self.server_name}] {tool.get('description', name)}"

            def make_caller(tool_name):
                def caller(**kwargs):
                    return self.call_tool(tool_name, kwargs)
                return caller

            tool_registry.register(
                name=prefixed_name,
                description=description,
                parameters=schema,
                fn=make_caller(name),
            )
        print(f"[MCP:{self.server_name}] Registered {len(self._tools)} tools into agent registry.")

    def disconnect(self):
        """Terminate the MCP server subprocess."""
        if self._process:
            try:
                self._process.terminate()
                self._process.wait(timeout=5)
            except Exception:
                self._process.kill()
            self._process = None
            print(f"[MCP:{self.server_name}] Disconnected.")


print("MCPClient class defined.")

In [ ]:
# --- Connect to Playwright MCP server (optional) ---
# Requires: npm install -g @playwright/mcp@latest
#
# The Playwright MCP server provides tools like:
#   browser_navigate, browser_click, browser_type, browser_snapshot,
#   browser_screenshot, browser_go_back, browser_go_forward, etc.

ENABLE_MCP_BROWSER = False  # Set to True to enable

mcp_client = None

def setup_mcp():
    global mcp_client
    if not ENABLE_MCP_BROWSER:
        print("MCP browser tools disabled. Set ENABLE_MCP_BROWSER = True to enable.")
        return

    mcp_client = MCPClient(
        server_command=["npx", "@playwright/mcp@latest"],
        server_name="playwright"
    )
    connected = mcp_client.connect()
    if connected:
        mcp_client.register_tools(registry)
        print(f"\nTotal tools available: {len(registry.list_tools())}")
    else:
        print("Continuing without browser tools.")

setup_mcp()

## 5. Custom Skills — Extend the Agent

Add your own tools and skills using decorators. Skills are higher-level capabilities that blend the three phases internally.

In [ ]:
# --- Example: Custom Skill ---

@registry.skill(name="summarize_file", description="Read a file and produce a short summary of its contents.")
def summarize_file(path: str) -> str:
    content = read_file(path)
    lines = content.strip().split("\n")
    total = len(lines)
    preview = "\n".join(lines[:20])
    return f"File: {path} ({total} lines)\nPreview (first 20 lines):\n{preview}"


@registry.skill(name="search_files", description="Search for files matching a glob pattern in a directory tree.")
def search_files(pattern: str, root: str = ".") -> str:
    import glob as globmod
    matches = globmod.glob(pattern, root_dir=root, recursive=True)
    if not matches:
        return f"No files matching '{pattern}' in {root}"
    return f"Found {len(matches)} matches:\n" + "\n".join(f"  {m}" for m in matches[:50])


@registry.tool(name="grep", description="Search for a regex pattern in files under a directory. Returns matching lines.")
def grep(pattern: str, path: str = ".", file_glob: str = "*") -> str:
    import glob as globmod, re
    files = globmod.glob(f"**/{file_glob}", root_dir=path, recursive=True)
    results = []
    for f in files[:100]:
        full = os.path.join(path, f)
        if not os.path.isfile(full):
            continue
        try:
            with open(full, "r", encoding="utf-8", errors="ignore") as fh:
                for i, line in enumerate(fh, 1):
                    if re.search(pattern, line):
                        results.append(f"{f}:{i}: {line.rstrip()}")
        except (PermissionError, OSError):
            continue
    if not results:
        return f"No matches for '{pattern}'"
    return f"{len(results)} matches:\n" + "\n".join(results[:30])


# --- Add your own tools/skills below! ---
# @registry.tool(name="my_tool", description="What it does")
# def my_tool(param: str) -> str:
#     return "result"


print(f"\nAll registered tools ({len(registry.list_tools())}):")
for t in registry.list_tools():
    label = "skill" if t.is_skill else "tool"
    print(f"  [{label:5s}] {t.name:20s} — {t.description[:60]}")

## 6. The Agent Loop

The core agentic loop. It sends the task to the LLM along with available tools, then iteratively:
1. **Gathers context** — the LLM reasons about what it needs
2. **Takes action** — calls tools to do work
3. **Verifies results** — inspects outputs and decides if done or needs more work

These phases blend naturally within each iteration.

In [ ]:
from openai import OpenAI


SYSTEM_PROMPT = """\
You are NSAgent, a capable task-execution agent. You solve tasks by working through
three blended phases:

1. GATHER CONTEXT — Understand what is needed. Read files, list directories, fetch web
   pages, or inspect the current state before acting.
2. TAKE ACTION — Use tools to accomplish the task. Write files, run commands, execute code,
   browse the web, etc.
3. VERIFY RESULTS — Check that your actions succeeded. Read back files you wrote, run tests,
   inspect outputs. If something is wrong, loop back to gather more context or take
   corrective action.

These phases blend together naturally. You may gather context and act in the same step,
or verify and gather more context if needed.

Guidelines:
- Be methodical. Understand before you act.
- Make the smallest changes needed to accomplish the task.
- Always verify your work before calling task_done.
- Use the task_done tool when the task is fully complete, with a summary of what you did.
- If you are stuck, explain what you tried and what went wrong.
"""


class Agent:
    """Agentic loop that coordinates LLM reasoning with tool execution."""

    def __init__(self, config: dict, tool_registry: ToolRegistry):
        self.config = config
        self.registry = tool_registry
        self.client = OpenAI(api_key=config["api_key"], base_url=config["base_url"])
        self.messages = []
        self.iteration = 0

    def run(self, task: str) -> str:
        """Execute a task through the agentic loop."""
        print(f"\n{'='*60}")
        print(f"TASK: {task}")
        print(f"{'='*60}\n")

        self.messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": task},
        ]
        self.iteration = 0

        while self.iteration < self.config["max_iterations"]:
            self.iteration += 1
            print(f"--- Iteration {self.iteration} ---")

            # Call LLM with tools
            try:
                response = self.client.chat.completions.create(
                    model=self.config["model"],
                    messages=self.messages,
                    tools=self.registry.to_openai_tools(),
                    temperature=self.config["temperature"],
                )
            except Exception as e:
                print(f"  LLM Error: {e}")
                return f"Agent failed: LLM error — {e}"

            choice = response.choices[0]
            message = choice.message

            # If the LLM has text content, display it
            if message.content:
                print(f"  Agent: {message.content[:200]}")

            # If no tool calls, the agent is done thinking
            if not message.tool_calls:
                self.messages.append({"role": "assistant", "content": message.content})
                print("\n[Agent finished — no more tool calls]")
                return message.content or "(no response)"

            # Process tool calls (blended phases)
            self.messages.append(message.model_dump())

            for tc in message.tool_calls:
                fn_name = tc.function.name
                try:
                    fn_args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    fn_args = {}

                # Determine phase label for display
                phase = self._classify_phase(fn_name)
                print(f"  [{phase}] {fn_name}({json.dumps(fn_args)[:80]})")

                # Execute the tool
                result = self.registry.call(fn_name, fn_args)

                # Check for task completion
                if result.startswith("TASK_COMPLETE:"):
                    print(f"\n✅ {result}")
                    self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
                    return result

                # Show truncated result
                preview = result[:150].replace("\n", " ") + ("..." if len(result) > 150 else "")
                print(f"    → {preview}")

                self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

        print(f"\n⚠️  Max iterations ({self.config['max_iterations']}) reached.")
        return "Agent stopped: max iterations reached."

    def _classify_phase(self, tool_name: str) -> str:
        """Heuristic to label which phase a tool call belongs to."""
        context_tools = {"read_file", "list_directory", "web_fetch", "search_files",
                         "grep", "summarize_file", "browser_snapshot"}
        verify_tools = {"task_done"}
        if tool_name in verify_tools:
            return "VERIFY "
        if tool_name in context_tools or tool_name.startswith("browser_snapshot"):
            return "CONTEXT"
        return "ACTION "


agent = Agent(CONFIG, registry)
print("Agent initialized and ready.")

## 7. Run the Agent

Give the agent a task and watch it work through the three phases.

In [ ]:
# --- Give the agent a task! ---

result = agent.run("""
List the files in the current directory and create a file called 'hello.txt' 
with the content 'Hello from NSAgent!'. Then read it back to verify it was written correctly.
""")

print(f"\n{'='*60}")
print(f"FINAL RESULT:\n{result}")

In [ ]:
# --- Interactive mode: enter your own task ---

# task = input("Enter a task for the agent: ")
# result = agent.run(task)
# print(f"\nRESULT: {result}")

## 8. Cleanup

In [ ]:
# Disconnect MCP server if connected
if mcp_client:
    mcp_client.disconnect()

print("Done.")